# Day 2｜张量操作与自动微分

> **来源**：d2l 2.1–2.5（张量 / 自动微分）｜李宏毅 ML 2021 Lec1–2｜3B1B 线代 Ep.1–4
> **目标**：把"张量形状"和"自动微分"练成肌肉记忆，它是后面所有模型的地基
> **前置**：无（这是起点）
> **复习状态**：首学 07-03 ｜ 已复习 0 次 ｜ 自测未做

---

## 0. 五秒回忆卡（闭卷先答）

1. 矩阵的每一列在几何上代表什么？
2. 为什么 `.backward()` 要求输出是标量？
3. 训练循环里忘了 `zero_grad()`，`optimizer.step()` 用的是谁的值？

---


## 1. 3B1B 线代：向量、矩阵与线性变换

### 1.1 向量是什么

- 别当成死板的数字表格，要想象成坐标系里有长度和方向的**箭头**。
- 空间里的每一个点，都可以看作是从原点射出的一个箭头。

### 1.2 基向量与线性组合

- $\hat{i}$ 和 $\hat{j}$ 是网格的横纵基本单位（向右 1 步、向上 1 步）。
- 任何向量不过是把这两个基本单位**拉伸缩短**再**相加**组合出来的结果。

### 1.3 矩阵就是"动词"

- 矩阵不是一堆数字，而是一次**对空间的变换**（拉扯、挤压、旋转）。
- 矩阵乘向量的画面：把这个向量扔进正在扭曲的空间里，看它最后落在哪。
- 矩阵的**列**记录的就是变换后 $\hat{i}$、$\hat{j}$ 的新落点。

### 1.4 矩阵乘法 = 连续的动作

- $AB$ 不是枯燥的算术题，而是**连续做两次空间变换**。
- 顺序从右往左读：先做 $B$ 变换，再做 $A$ 变换（先穿内衣，再穿外套）。
- 直接推论：矩阵乘法**不满足交换律**，$AB \neq BA$。

---


## 2. 机器学习：找函数三步曲

**核心思想**：机器学习就是让机器自己去找一个最合适的函数。

### 2.1 三步曲（以预测频道播放量为例）

| 步骤 | 做什么 | 谁的工作 |
| --- | --- | --- |
| 1. 定义模型 | 先猜一个函数框架，如 $y = wx + b$ | 人设计结构，机器负责找 $w, b$ |
| 2. 定义损失 | 设定一个衡量"有多烂"的指标 | 人 |
| 3. 优化参数 | 梯度下降，让 Loss 变小 | 机器 |

- $x$ 是已知特征，$w$ 是权重，$b$ 是偏置，$w$ 和 $b$ 就是要找的未知参数。
- **梯度下降大白话**：蒙着眼睛下山。算出脚底的坡度（梯度），朝低的方向迈一步，反复迭代到谷底。

### 2.2 从机器学习到深度学习

- **线性模型的局限（Model Bias）**：直线就是直线，再怎么调也拟合不出复杂曲线。
- **破局之法**：把许多个简单非线性函数（Sigmoid / ReLU）像拼乐高一样组合起来，逼近任意复杂曲线。
- 把这些函数一层层叠得很深、参数量很大，就是**神经网络**，也就是**深度学习**。

---


## 3. 张量操作（PyTorch 基础）

> 阅读约定：每段代码上方的注释写目的，下面记 **shape 变化**。

| 主题 | API | 关键点 |
| --- | --- | --- |
| 创建 | `arange` / `zeros` / `ones` / `randn` / `tensor` | `randn` 是标准正态；`tensor` 从列表造 |
| 形状 | `shape` / `numel` / `reshape` | `-1` 表示自动推导该维长度 |
| 运算 | `+ - * / **` / `exp` | 都是逐元素；矩阵乘要写 `@` 或 `matmul` |
| 拼接 | `cat((A,B), dim)` | `dim=0` 竖直拼、`dim=1` 水平拼 |
| 广播 | 自动生效 | 从右往左对齐，只有维度为 1 或缺失才能广播 |
| 就地修改 | `X[:] = ...` / `+=` | 保持内存地址不变；`Y = Y + X` 会新建对象 |

---


In [1]:
# pytorch 导入的是 torch
import torch

# 用 arange 创建一个行向量 x 包含以0开始的前12个整数
x = torch.arange(12)
print(x)

# 可以通过张量的 shape 属性来访问张量的形状
print(x.shape)

# 元素个数
print(x.numel())

# 改变已有张量的形状
X = x.reshape(3, 4)
print(X)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
torch.Size([12])
12
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


In [2]:
import torch

# 设置一些全0的张量
torch.zeros((2, 3, 4))

tensor([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]])

In [3]:
import torch

# 设置一些全1的张量
torch.ones((2, 3, 4))

tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])

In [4]:
# 创建一定形状的随机正态分布张量
import torch
torch.randn(3, 4)

tensor([[ 0.0186, -1.0403, -0.9883,  0.1590],
        [-1.0471, -1.2213, -1.6724,  1.3010],
        [ 1.5608,  1.6532, -0.7464, -0.2534]])

In [5]:
# 嵌套列表
import torch
torch.tensor([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

tensor([[2, 1, 4, 3],
        [1, 2, 3, 4],
        [4, 3, 2, 1]])

In [6]:
# 一些基本的运算
import torch
x = torch.tensor([1.0, 2, 4, 8])
y = torch.tensor([2, 2, 2, 2])
x + y, x - y, x * y, x / y, x ** y

(tensor([ 3.,  4.,  6., 10.]),
 tensor([-1.,  0.,  2.,  6.]),
 tensor([ 2.,  4.,  8., 16.]),
 tensor([0.5000, 1.0000, 2.0000, 4.0000]),
 tensor([ 1.,  4., 16., 64.]))

In [7]:
# 求幂 e 的次方
x = torch.tensor([1.0, 2, 4, 8])
torch.exp(x)

tensor([2.7183e+00, 7.3891e+00, 5.4598e+01, 2.9810e+03])

In [8]:
# 将张量连结在一起形成大的张量
import torch
X = torch.arange(12, dtype=torch.float32).reshape((3, 4))
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2 ,3, 4], [4, 3, 2, 1]])
torch.cat((X, Y), dim=0), torch.cat((X, Y), dim=1)

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.],
         [ 2.,  1.,  4.,  3.],
         [ 1.,  2.,  3.,  4.],
         [ 4.,  3.,  2.,  1.]]),
 tensor([[ 0.,  1.,  2.,  3.,  2.,  1.,  4.,  3.],
         [ 4.,  5.,  6.,  7.,  1.,  2.,  3.,  4.],
         [ 8.,  9., 10., 11.,  4.,  3.,  2.,  1.]]))

In [9]:
# 逻辑运算
import torch
X = torch.arange(12, dtype=torch.float32).reshape((3, 4))
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2 ,3, 4], [4, 3, 2, 1]])
X == Y

tensor([[False,  True, False,  True],
        [False, False, False, False],
        [False, False, False, False]])

In [10]:
# 张量中全部元素求和
import torch
X = torch.arange(12, dtype=torch.float32).reshape((3, 4))
X.sum()

tensor(66.)

In [11]:
# 广播机制
import torch
a = torch.arange(3).reshape((3, 1))
b = torch.arange(2).reshape((1, 2))
a, b, a + b

(tensor([[0],
         [1],
         [2]]),
 tensor([[0, 1]]),
 tensor([[0, 1],
         [1, 2],
         [2, 3]]))

In [12]:
# 索引与切片
import torch
X = torch.arange(12, dtype=torch.float32).reshape((3, 4))
X[-1], X[1:3]

(tensor([ 8.,  9., 10., 11.]),
 tensor([[ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]))

In [13]:
# 通过指定索引将元素写入矩阵
import torch
X = torch.arange(12, dtype=torch.float32).reshape((3, 4))
X[1, 2] = 9
X

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  9.,  7.],
        [ 8.,  9., 10., 11.]])

In [14]:
# 批量为多个元素赋值相同的值
import torch
X = torch.arange(12, dtype=torch.float32).reshape((3, 4))
X[0:2, :] = 12
X

tensor([[12., 12., 12., 12.],
        [12., 12., 12., 12.],
        [ 8.,  9., 10., 11.]])

In [15]:

# 原地分配: X += Y 或者 Z[:] = X + Y（ Y = X + Y 不是）
before = id(Y)
Y = Y + X
id(Y) == before
Z = torch.zeros_like(Y)
print('id(Z):', id(Z))
Z[:] = X + Y
print('id(Z):', id(Z))

id(Z): 13003621056
id(Z): 13003621056


In [16]:
# 转换为其他 Python 对象
A = X.numpy()
B = torch.tensor(A)
type(A), type(B)

(numpy.ndarray, torch.Tensor)

In [17]:
# 将大小为1的张量转化为 Python 标量
a = torch.tensor([3.5])
a, a.item(), float(a), int(a)

(tensor([3.5000]), 3.5, 3.5, 3)

## 4. 自动微分（Autograd）

- `requires_grad=True` 之后，PyTorch 会把这条计算路径记进**计算图**。
- `.backward()` 从标量出发，沿链式法则把梯度回填到各个叶子张量的 `.grad`。
- 叶子张量的 `.grad` **默认是 `None`，而且是累加的**。

---


In [18]:
# 自动微分的一个简单的例子
import torch

x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

In [19]:
x.requires_grad_(True) # 等价于 x = torch.arange(4.0, requires_grad=True)
x.grad # 默认是 None

y = 2 * torch.dot(x, x)
y

tensor(28., grad_fn=<MulBackward0>)

In [20]:
y.backward()
x.grad

tensor([ 0.,  4.,  8., 12.])

In [21]:
x.grad == 4 * x

tensor([True, True, True, True])

In [22]:
# y = x.sum() 求导
x.grad.zero_() # 先清空
y = x.sum()
y.backward()
x.grad

tensor([1., 1., 1., 1.])

## 5. 非标量反向传播

**核心法则**：`.backward()` 需要一个标量起点——链式法则的种子就是 $\partial L / \partial L = 1$。

如果输出 `y` 是向量或矩阵，两种正确写法：

| 写法 | 得到什么 | 用途 |
| --- | --- | --- |
| `y.sum().backward()`（或 `.mean()`） | 各分量导数之和，标量 | 训练里的默认做法 |
| `y.backward(torch.ones_like(x))` | 等价于对每个分量求导再求和；把 `ones` 换成 one-hot，就能逐个取出 Jacobian 的每一列 | 求 Jacobian、高阶导、方向导数（VJP） |

> 更正旧笔记：把 `backward(grad_tensors)` 说成"实战极少用的底层语法"是不准确的。
> 它是求 Jacobian / 高阶导的标准入口；只是在常规训练里，用 `sum()` 更省事。

---


In [23]:
x.grad.zero_()
y = x * x # 并非点积
y.sum().backward()
x.grad

tensor([0., 2., 4., 6.])

## 6. detach：切断梯度

`.detach()` 是"失忆魔法"，也是"剪断求导电线"：把一个经过复杂运算的变量，强行变成没有背景故事的**常数**。

- **效果**：反向传播走到被 detach 的变量就停住，不再往前传。
- **典型场景**：把带梯度的张量转成 numpy 或画图（`.detach().numpy()`）；把某个中间结果当常数用。
- **和 `requires_grad=False` 的区别**：后者是"这个参数不参与训练"，前者是"这条计算路径到此为止"。

---


In [24]:
# 如果你正常对 z 呼叫反向传播 z.backward()，PyTorch 会顺藤摸瓜，用复合导数的链式法则，算出 z 对 x 的导数是 3x^2。
# 但是，如果你突然中二病发作，想跟 PyTorch 说：“在算 z 的时候，请把 y 当作一个毫无感情的数字常数，忘记它是怎么从 x 算出来的！” 这时候，就需要用到 .detach() 这个“失忆魔法”了。
x.grad.zero_()
y = x * x
u = y.detach()
z = u * x

z.sum().backward()
x.grad == u

tensor([True, True, True, True])

## 7. 动态计算图

- PyTorch 的图是"边跑边画"的：前向执行到哪，就把哪条路径记下来。
- 就算代码里写满 `if` / `else` / `while` / `for`，甚至每次循环次数都不一样，PyTorch 也能精准记录**当次执行的真实轨迹**。
- **威力**：只要函数本身分段连续，就可以放心在模型里用 Python 原生控制流，`.backward()` 不会算错方向。

---


In [25]:
import torch

# 1. 定义一个带有复杂 if 和 while 的函数
def f(a):
    b = a * 2
    # 只要 b 的绝对值（范数）不到 1000，就一直翻倍
    while b.norm() < 1000:
        b = b * 2
    # 分支判断
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

# 2. 随机生成一个标量 a，并开启梯度追踪
a = torch.randn(size=(), requires_grad=True)
print("随机生成的输入 a:", a.item())

# 3. 前向传播：把 a 扔进复杂的函数里算出 d
d = f(a)
print("计算出的结果 d:", d.item())

# 4. 反向传播：自动求导
d.backward()
print("PyTorch 算出的梯度 a.grad:", a.grad.item())
print("数学推导的真实梯度 d / a:", (d / a).item())

# 5. 终极验证：它们相等吗？
print("梯度计算是否完全正确？", a.grad == d / a)

随机生成的输入 a: 0.22767148911952972
计算出的结果 d: 1865.0848388671875
PyTorch 算出的梯度 a.grad: 8192.0
数学推导的真实梯度 d / a: 8192.0
梯度计算是否完全正确？ tensor(True)


## 8. 梯度累加陷阱

**核心概念**：PyTorch 的梯度默认**累加**，不会自动覆盖。

**场景**：$y = 2x^2$，导数是 $4x$，取 $x = 3$。

- 第一次 `y.backward()`：算出 `12.0`，写进 `.grad`。
- 不清零直接再跑一次：又算出 `12.0`，**加到**原来的记录上。
- 结果：`x.grad = 24.0`（是加法，不是乘法）。

**为什么危险**：训练循环里 `optimizer.step()` 用的就是 `.grad` 当前的值。忘记清零等于用的是"历史所有批次梯度之和"——等效学习率被放大、更新方向被旧数据污染，Loss 从抖动走向发散。

**黄金法则**：每次计算新梯度前先 `x.grad.zero_()`（实战里就是 `optimizer.zero_grad()`）。

---


## 9. 自测（3+1）

1. **概念**：为什么 `.backward()` 要求标量输出？两种替代写法各自的用途是什么？
2. **计算**：`a = arange(3).reshape(3,1)`、`b = arange(2).reshape(1,2)`，`a + b` 的结果矩阵是什么？
3. **计算**：`x = [0,1,2,3]`、`y = 2 * x * x`（逐元素），`y.sum().backward()` 之后 `x.grad` 是多少？不清零再跑一次呢？
4. **编程**：写一个含 `while` + `if`、返回标量的 `f(a)`，用 `torch.autograd.gradcheck` 验证它的梯度。
